In [ ]:
!pip install mlflow

In [ ]:

# Test the current directory and MLFlow directory and package
import os
print(os.getcwd())

import mlflow
print("Tracking URI:", mlflow.get_tracking_uri())

In [ ]:
# Task 1

In [ ]:
import mlflow
import mlflow.sklearn

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Dataset
X, y = load_iris(return_X_y=True)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Parameters to track
n_estimators = 2
max_depth = 3

mlflow.set_experiment("Simple_Iris_Classification 1")

with mlflow.start_run():

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # Log parameters and metrics
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("accuracy", accuracy)

    # Log model with trusted types specified for skops
    mlflow.sklearn.log_model(
        model,
        "random_forest_model 1",
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

    print("Accuracy:", accuracy)

In [ ]:
# Parameters to track
n_estimators = 15
max_depth = 4

mlflow.set_experiment("Simple_Iris_Classification 2")

with mlflow.start_run():

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # Log parameters and metrics
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("accuracy", accuracy)

    # Log model with trusted types specified for skops
    mlflow.sklearn.log_model(
        model,
        "random_forest_model 2",
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

    print("Accuracy:", accuracy)

In [ ]:
# Parameters to track
n_estimators = 15
max_depth = 4

mlflow.set_experiment("Simple_Iris_Classification 3")

with mlflow.start_run():

    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    # Log parameters and metrics
    mlflow.log_param("n_estimators", n_estimators)
    mlflow.log_param("max_depth", max_depth)
    mlflow.log_metric("accuracy", accuracy)

    # Log model with trusted types specified for skops
    mlflow.sklearn.log_model(
        model,
        "random_forest_model 3",
        skops_trusted_types=["sklearn.tree._tree.Tree"]
    )

    print("Accuracy:", accuracy)

In [ ]:
# model 1: n_estimators=2,max_depth=3
# model 2: n_estimators=15, max_depth=3

In [ ]:
import mlflow
import mlflow.sklearn
import numpy as np
import time

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

# Connect to your MLflow server
mlflow.set_tracking_uri("http://127.0.0.1:5000")

# Create or select experiment
mlflow.set_experiment("Iris_Model_Efficiency")

# Load Iris dataset
iris = load_iris()
X = iris.data
y = iris.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2,
    random_state=42, stratify=y
)

# Train and log two models
for n in [2, 15]:

    with mlflow.start_run(run_name=f"RF_{n}_trees"):

        model = RandomForestClassifier(
            n_estimators=n,
            max_depth=4,
            random_state=42
        )

        start = time.perf_counter()
        model.fit(X_train, y_train)
        training_time = time.perf_counter() - start

        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        macro_f1 = f1_score(
            y_test, predictions, average="macro"
        )

        # Measure prediction latency
        times = []
        for _ in range(100):
            start = time.perf_counter()
            model.predict(X_test)
            times.append(
                (time.perf_counter() - start) * 1000
            )

        p50 = np.percentile(times, 50)
        p95 = np.percentile(times, 95)

        mlflow.log_param("n_estimators", n)
        mlflow.log_param("max_depth", 4)

        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("macro_f1", macro_f1)
        mlflow.log_metric("training_time_sec", training_time)
        mlflow.log_metric("prediction_p50_ms", p50)
        mlflow.log_metric("prediction_p95_ms", p95)

        # Fix: explicitly trust the sklearn tree structure type for skops
        mlflow.sklearn.log_model(
            model, 
            name="iris_model",
            skops_trusted_types=["sklearn.tree._tree.Tree"]
        )

        print(f"{n} trees:")
        print(f"Accuracy: {accuracy:.4f}")
        print(f"Macro F1: {macro_f1:.4f}")
        print(f"Training time: {training_time:.6f} sec")
        print(f"Prediction p50: {p50:.4f} ms")
        print(f"Prediction p95: {p95:.4f} ms")
        print()

In [ ]:
# Task 2

In [ ]:
%pip install mlflow scikit-learn pandas joblib

In [ ]:
from pathlib import Path
from time import perf_counter
from tempfile import TemporaryDirectory

import joblib
import numpy as np
import pandas as pd
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from mlflow.models import infer_signature

# Read your supplied dataset.
data = pd.read_csv("Iris.csv")

features = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm"
]

X = data[features]       # Exclude Id: it is only a row identifier.
y = data["Species"]

# Use identical data and random seeds for both configurations.
# This is a validation comparison; use a separate test set
# for a final assessment after choosing the configuration.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Store experiments locally; routine logging needs no Wi-Fi.
mlflow.set_tracking_uri("sqlite:///iris_mlflow.db")
mlflow.set_experiment("Iris_Model_Efficiency")

results = []

for trees in [2, 15]:
    with mlflow.start_run(run_name=f"RF_{trees}_trees_depth_4"):

        model = RandomForestClassifier(
            n_estimators=trees,
            max_depth=4,
            random_state=42,
            n_jobs=1
        )

        # Measure training time.
        start = perf_counter()
        model.fit(X_train, y_train)
        training_ms = (perf_counter() - start) * 1000

        predictions = model.predict(X_valid)

        accuracy = accuracy_score(y_valid, predictions)
        macro_f1 = f1_score(
            y_valid, predictions, average="macro"
        )

        # Benchmark one flower per prediction.
        sample = X_valid.iloc[[0]]

        # Warm up before measuring.
        for _ in range(20):
            model.predict(sample)

        timings_ms = []
        for _ in range(300):
            start = perf_counter()
            model.predict(sample)
            timings_ms.append(
                (perf_counter() - start) * 1000
            )

        # Measure an uncompressed deployment model file.
        with TemporaryDirectory() as folder:
            model_path = Path(folder) / "iris_model.joblib"
            joblib.dump(model, model_path, compress=0)
            model_size_kb = model_path.stat().st_size / 1024

            mlflow.log_artifact(
                str(model_path),
                artifact_path="deployment_model"
            )

        mlflow.log_params({
            "n_estimators": trees,
            "max_depth": 4,
            "random_state": 42,
            "n_jobs": 1,
            "validation_fraction": 0.2,
            "training_samples": len(X_train),
            "validation_samples": len(X_valid),
            "benchmark_repeats": 300
        })

        metrics = {
            "validation_accuracy": accuracy,
            "validation_macro_f1": macro_f1,
            "training_time_ms": training_ms,
            "prediction_p50_ms": float(
                np.percentile(timings_ms, 50)
            ),
            "prediction_p95_ms": float(
                np.percentile(timings_ms, 95)
            ),
            "model_size_kb": model_size_kb
        }

        mlflow.log_metrics(metrics)

        # Save the model with its input/output structure.
        mlflow.sklearn.log_model(
            sk_model=model,
            name="iris_random_forest",
            serialization_format="cloudpickle",
            signature=infer_signature(X_train, model.predict(X_train)),
            input_example=X_train.head(2)
        )

        results.append({
            "n_estimators": trees,
            **metrics
        })

comparison = pd.DataFrame(results)
print(comparison.round(4).to_string(index=False))

In [ ]:
import mlflow
mlflow.set_experiment("Iris _Model_Efficiency")

In [ ]:
# Task 3

In [ ]:
%pip install mlflow scikit-learn pandas cloudpickle

In [ ]:
from pathlib import Path
import hashlib
import platform

import pandas as pd
import sklearn
import mlflow
import mlflow.sklearn

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from mlflow.models import infer_signature

# Local database supports tracking and model registration.
mlflow.set_tracking_uri("sqlite:///iris_mlflow.db")
experiment = mlflow.set_experiment("Iris_Task_3")

# Read the supplied dataset.
csv_path = Path("Iris.csv")
data = pd.read_csv(csv_path)

features = [
    "SepalLengthCm",
    "SepalWidthCm",
    "PetalLengthCm",
    "PetalWidthCm"
]

X = data[features]     # Exclude the Id column.
y = data["Species"]

# Fixed split for a reproducible validation comparison.
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# Fingerprint identifies the exact dataset file used.
dataset_hash = hashlib.sha256(
    csv_path.read_bytes()
).hexdigest()

# Retain the split membership for reproduction.
split_manifest = {
    "training_row_indices": X_train.index.tolist(),
    "validation_row_indices": X_valid.index.tolist()
}

run_ids = []

for trees in [2, 15]:
    with mlflow.start_run(
        run_name=f"RF_{trees}_trees_depth_4"
    ) as run:

        model = RandomForestClassifier(
            n_estimators=trees,
            max_depth=4,
            random_state=42,
            n_jobs=1
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_valid)

        # TASK 3A: Track experiment parameters.
        mlflow.log_params({
            "n_estimators": trees,
            "max_depth": 4,
            "random_state": 42,
            "n_jobs": 1,
            "validation_fraction": 0.2
        })

        # TASK 3B: Record comparable evaluation metrics.
        mlflow.log_metrics({
            "validation_accuracy": accuracy_score(
                y_valid, predictions
            ),
            "validation_macro_f1": f1_score(
                y_valid, predictions, average="macro"
            )
        })

        # TASK 3D: Preserve reproducibility information.
        mlflow.set_tags({
            "dataset": "Iris.csv",
            "dataset_sha256": dataset_hash,
            "python_version": platform.python_version(),
            "sklearn_version": sklearn.__version__,
            "mlflow_version": mlflow.__version__,
            "evaluation_type": "validation"
        })

        mlflow.log_artifact(
            str(csv_path),
            artifact_path="dataset"
        )

        mlflow.log_dict(
            split_manifest,
            "reproducibility/data_split.json"
        )

        mlflow.log_dict(
            {
                "features": features,
                "target": "Species",
                "excluded_column": "Id",
                "preprocessing": "No scaling or imputation"
            },
            "reproducibility/input_configuration.json"
        )

        # Preserve predictions as evidence.
        prediction_table = X_valid.copy()
        prediction_table["actual_species"] = y_valid
        prediction_table["predicted_species"] = predictions

        mlflow.log_table(
            prediction_table,
            artifact_file="validation_predictions.json"
        )

        # TASK 3C: Save and register a model version.
        model_info = mlflow.sklearn.log_model(
            sk_model=model,
            name="iris_random_forest",
            registered_model_name="IrisRandomForest",
            serialization_format="cloudpickle",
            signature=infer_signature(
                X_train, model.predict(X_train)
            ),
            input_example=X_train.head(2),
            pip_requirements=[
                f"scikit-learn=={sklearn.__version__}",
                f"pandas=={pd.__version__}",
                f"cloudpickle=={__import__('cloudpickle').__version__}"
            ]
        )

        run_ids.append(run.info.run_id)

        print(
            f"Trees: {trees} | "
            f"Run ID: {run.info.run_id} | "
            f"Model: {model_info.model_uri}"
        )

# Compare only the runs created by this execution.
runs = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id]
)
comparison = runs[runs["run_id"].isin(run_ids)][[
    "run_id",
    "params.n_estimators",
    "params.max_depth",
    "metrics.validation_accuracy",
    "metrics.validation_macro_f1"
]]

print("\nExperiment comparison:")
print(comparison.to_string(index=False))

In [ ]:
from mlflow import MlflowClient

client = MlflowClient()

VERSION_TO_USE = "2"  # Replace with your reviewed version.

client.set_registered_model_alias(
    name="IrisRandomForest",
    alias="candidate",
    version=VERSION_TO_USE
)

# Load that specific version for reproducible inference.
loaded_model = mlflow.sklearn.load_model(
    f"models:/IrisRandomForest/{VERSION_TO_USE}"
)

sample = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=features
)

print("Predicted species:", loaded_model.predict(sample)[0])

In [ ]:
# Task 4

Run the following three cells in order. They save a real model, register its returned URI, and test the offline model. The deployment configuration uses 15 trees and maximum depth 4. The displayed accuracy is for this fixed split; repeated model selection requires a separate final test set.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
import mlflow
import mlflow.sklearn
from mlflow.tracking import MlflowClient
from mlflow.models import infer_signature
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# Run this cell before the registration cell below.
mlflow.set_tracking_uri("sqlite:///iris_mlflow.db")
mlflow.set_experiment("Iris_Task_4_Deployment")
features = ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]
csv_path = Path("Iris.csv")
if csv_path.exists():
    data = pd.read_csv(csv_path)
    X = data[features]
    y = data["Species"]
    dataset_source = str(csv_path)
else:
    # Allows Task 4 to run independently when Iris.csv is unavailable.
    iris = load_iris()
    X = pd.DataFrame(iris.data, columns=features)
    y = pd.Series(["Iris-" + iris.target_names[i] for i in iris.target])
    dataset_source = "sklearn.datasets.load_iris"

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
local_model_path = Path("iris_deployment") / "iris_model.joblib"
local_model_path.parent.mkdir(parents=True, exist_ok=True)

with mlflow.start_run(run_name="RF_15_trees_depth_4_deployment") as run:
    run_id = run.info.run_id
    deployment_model = RandomForestClassifier(
        n_estimators=15, max_depth=4, random_state=42, n_jobs=1
    )
    deployment_model.fit(X_train, y_train)
    test_predictions = deployment_model.predict(X_test)
    test_accuracy = accuracy_score(y_test, test_predictions)
    mlflow.log_params(deployment_model.get_params())
    mlflow.set_tag("dataset_source", dataset_source)
    mlflow.log_metric("test_accuracy", test_accuracy)
    deployment_model_info = mlflow.sklearn.log_model(
        sk_model=deployment_model,
        artifact_path="iris_deployment_model",
        serialization_format="cloudpickle",
        signature=infer_signature(X_train, deployment_model.predict(X_train)),
        input_example=X_train.head(2)
    )
    # Preserve the actual logged URI; no run ID needs to be pasted manually.
    deployment_model_uri = deployment_model_info.model_uri
    joblib.dump(deployment_model, local_model_path)
    mlflow.log_artifact(str(local_model_path), artifact_path="offline_deployment")

print("Run ID:", run_id)
print("Logged model:", deployment_model_uri)
print("Offline model saved to:", local_model_path.resolve())
print("Test accuracy:", test_accuracy)


In [ ]:
# Register the model saved by the previous cell.
registered = mlflow.register_model(
    model_uri=deployment_model_uri,
    name="IrisDeploymentModel"
)
client = MlflowClient()
client.set_registered_model_alias(
    name="IrisDeploymentModel", alias="candidate", version=registered.version
)
print("Registered version:", registered.version)
print("Candidate alias:", client.get_model_version_by_alias(
    "IrisDeploymentModel", "candidate"
).version)


In [ ]:
# Offline prediction: this uses the saved file and needs no MLflow server.
offline_model = joblib.load(local_model_path)
flower = pd.DataFrame([[5.1, 3.5, 1.4, 0.2]], columns=features)
print("Predicted species:", offline_model.predict(flower)[0])
